In [1]:
import pandas as pd
import csv
import pickle
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import os
import random
from itertools import product


from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search

In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx
# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()
random.seed(42)

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each (or use len(...) for "all")
num_topics = len(topic_cols)  # e.g., 100
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [10]:
import pandas as pd
import statsmodels.api as sm

def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

# usage
X = to_ar1_innovations(X)


In [ ]:
# Run a loop to estimate grid, find best parameters, estimate grid around best parameters, repeat

# Objective Function
def objective_function(row):
    if 1.96 < row['kappa_tstat'] <= 20:
        return row['r2_insample_stage2']
    else:
        return -float('inf')


# number of loops
refinement_rounds = 20

# Initial grid 
window_sizes = [10,50,100,200,300,400]
n_lags = [1,3,4,5,7,8,10]
lambda_base = 0.001
lambda_values = [lambda_base * factor for factor in [0.5, 0.8, 0.9, 1.0, 1.1, 1.2, 1.5]]

# IMPORTANT: match grid_search expectations: window_sizes, n_lags, lambdas
current_param_grid = {
    'window_sizes': window_sizes,
    'n_lags': n_lags,
    'lambdas': lambda_values
}

results_all_rounds = []

# ----------------------------------------
# Loop
# ----------------------------------------
for i in range(refinement_rounds):

    print(f"\n🔷 Starting Refinement Round {i+1}")

    # Run grid search
    summary_df, coefficients_df = grid_search(
        X, y, current_param_grid, verbose=True
    )

    # Evaluate objective
    summary_df['objective'] = summary_df.apply(objective_function, axis=1)

    # Track results
    results_all_rounds.append(summary_df.copy())

    # Pick best point
    best_row = summary_df.loc[summary_df['objective'].idxmax()]
    best_window = int(best_row['window_size'])
    best_n_lags = int(best_row['n_lags'])
    best_lambda = float(best_row['lambda'])

    print(f"Best so far (round {i+1}):")
    print(best_row[['window_size', 'n_lags', 'lambda', 'objective']])

    # ----------------------------------------
    # build new grid around best point
    # ----------------------------------------

    # shrink window search range
    window_sizes_refined = list(range(best_window - 25, best_window + 26, 5))
    window_sizes_refined = [w for w in window_sizes_refined if w > 20]

    # shrink n_lags search range
    n_lags_refined = list(range(best_n_lags - 1, best_n_lags + 2))
    n_lags_refined = [l for l in n_lags_refined if l >= 1]

    # shrink lambda range (currently ±20% around best)
    lambda_values_refined = np.linspace(
        0.8 * best_lambda,
        1.2 * best_lambda,
        num=10
    )

    # update grid for next iteration
    # AGAIN: keep keys consistent with grid_search: window_sizes, n_lags, lambdas
    current_param_grid = {
        'window_sizes': window_sizes_refined,
        'n_lags': n_lags_refined,
        'lambdas': lambda_values_refined
    }


# -------------------------------------------------
# 🏆 Final Result
# -------------------------------------------------
final_df = pd.concat(results_all_rounds, ignore_index=True)
final_df['objective'] = final_df.apply(objective_function, axis=1)
best_overall = final_df.loc[final_df['objective'].idxmax()]

print("\n🏆 Best Hyperparameters After Refinement:")
print(best_overall[['window_size', 'n_lags', 'lambda', 'objective']])

print("\n🏆 Best Hyperparameters After Refinement:")
print(best_overall[['window_size','n_lags','lambda','objective']])



🔷 Starting Refinement Round 1
Testing 294 configurations...


Grid search:  19%|█▉        | 56/294 [02:04<11:28,  2.89s/it]

In [10]:
# print for the 5 rows with the highest objective values r2 insample_stage2 and stage1, kappa ,kappa_tstat, lambda, window_size, n_lags
top_10 = final_df.nlargest(10, 'objective')
print(top_10[['r2_insample_stage2', 'r2_insample_stage1', 'r2_oos_stage2', 'kappa', 'kappa_tstat', 'lambda', 'window_size', 'n_lags']])

     r2_insample_stage2  r2_insample_stage1  r2_oos_stage2     kappa  \
621            0.005300            0.017287       0.001700  0.449607   
279            0.005280            0.020062       0.002025  0.430088   
293            0.004849            0.014511       0.000854  0.460478   
679            0.004589            0.012669       0.000068  0.470522   
618            0.005123            0.023192       0.001862  0.407079   
350            0.004192            0.010221      -0.000634  0.485122   
751            0.003869            0.009019      -0.000876  0.490253   
597            0.004663            0.026790      -0.003545  0.394399   
437            0.003417            0.006916      -0.001228  0.506163   
287            0.004551            0.027160       0.001227  0.370665   

     kappa_tstat    lambda  window_size  n_lags  
621     5.440399  0.002290          195       5  
279     5.243908  0.002240          195       5  
293     5.307970  0.002347          195       5  
679    

In [6]:
final_df2 = pd.read_parquet('grid_search_optimization.parquet')

In [7]:
# save
final_df.to_parquet('grid_search_optimization.parquet', index=False)

In [8]:
summary_df.to_parquet('grid_search_summary_9.parquet', index=False)


In [9]:
coefficients_df.to_parquet('grid_search_coefficients_9.parquet', index=False)